# Video -> walkable world (max quality, `ultra`)

Upload ONE video to Drive; this notebook produces a max-quality 3DGS splat **and** a
gravity-aligned **walkable world** bundle back on Drive. Runtime: **A100 GPU**. Honest
time estimate: **~5-8 h** for a ~2 min 1080p video at the `ultra` preset — the COLMAP
mapper and the 120k training iters dominate the wall clock.

It sidesteps the two traps of the naive preset video path: **frame explosion** (premium's
`fps=30` turns a 2-min clip into ~3750 frames and the CPU COLMAP mapper dies — here you set
a low `EXTRACT_FPS` for ~300-800 frames) and **silent upscaling** (the preset scale filter
force-resizes the long edge *up* even when the source is smaller — here frames are extracted
at NATIVE resolution and fed as a photo set, which the pipeline uses as-is).

## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Clone repo (branch `chore/strip-to-core`)

In [ ]:
import os
REPO_URL = "https://github.com/mehmettahacumurcu/gaussian-splatter.git"
BRANCH   = "chore/strip-to-core"   # Phase 2 work lives here, NOT main
REPO_DIR = "/content/gaussian-splatter"
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull   # re-run picks up fixes pushed since the first clone
%cd {REPO_DIR}
!git log --oneline -1

## 2. Mount Drive early

The verification notebooks mount Drive *last*; this one mounts it **first**, on purpose.
Two reasons: the **input video lives on Drive** (the CONFIG cell reads it a few cells down),
and mounting *before* bootstrap lets `--colmap-cuda` **restore its prebuilt COLMAP env**
from the Drive tarball (`MyDrive/4dgs/colmap-cuda/colmap-env.tar.gz`) instead of doing a
fresh ~1-2 min micromamba solve. `drive.mount` is idempotent, so the save cell at the end
just calls the helpers — no remount needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Environment (deps + gsplat + CUDA COLMAP)

`--colmap-cuda` installs a headless GPU-SIFT COLMAP wrapper at `/usr/local/bin/colmap`
(restored from the Drive tarball mounted above, or a fresh micromamba solve). If it fails it
removes the wrapper and falls back to the apt CPU build — the training cell detects the
wrapper's absence and adds `--colmap-cpu` automatically.

In [ ]:
!bash colab/bootstrap.sh --colmap-cuda

In [ ]:
# If this errors about numpy: Runtime -> Restart session, then re-run THIS cell only.
# (The chdir guard below re-enters the repo after a restart resets cwd to /content.)
import os
if os.path.isdir("/content/gaussian-splatter"):
    os.chdir("/content/gaussian-splatter")
import torch, gsplat
print("torch", torch.__version__, "| gsplat", gsplat.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 4. Config — edit these

Everything you tune lives in one cell. `VIDEO_PATH` must point at your uploaded video on
Drive. `EXTRACT_FPS` controls how many frames get pulled (the extraction cell sanity-checks
the count). `RUN_EVAL` decides whether every 8th frame is held out for metrics.

In [ ]:
# ---- all user knobs live here -------------------------------------------
VIDEO_PATH  = "/content/drive/MyDrive/room_render.mp4"  # <-- EDIT: your video on Drive
SCENE       = "myroom-max"        # data/<SCENE>/ workdir + Drive output folder name
EXTRACT_FPS = 4                   # frames/sec to pull. ~500 frames from a 2-min clip.
                                  # Keep TOTAL frames in the 300-800 band (next cell warns):
                                  # too few = coverage holes, too many = slow CPU mapper.
PRESET      = "ultra"             # max-quality preset (see scripts/static_3dgs.py)
RUN_EVAL    = False               # False = train on ALL frames (best final quality --
                                  #         "keeper" runs; no held-out metrics printed).
                                  # True  = hold out every 8th frame for PSNR/SSIM/LPIPS
                                  #         (comparable numbers, slightly fewer train views).
print(f"scene={SCENE}  preset={PRESET}  fps={EXTRACT_FPS}  eval={RUN_EVAL}")
print(f"video={VIDEO_PATH}")

## 5. Extract frames at native resolution

Copies the video to the local VM disk first (decoding straight off the Drive FUSE mount is a
network round-trip per read — the same ~0.2 it/s lesson that bans training through Drive),
then runs `ffmpeg` with an **fps filter only** — no scale filter, so frames come out at the
source resolution (no upscaling, no aspect stretch). Idempotent: it skips extraction if
`data/<SCENE>/images/` already has frames. The frame count is sanity-checked below.

In [ ]:
import os, glob, shutil, subprocess

assert os.path.exists(VIDEO_PATH), (
    f"VIDEO_PATH not found: {VIDEO_PATH}\n"
    f"  -> upload your video to Drive and set VIDEO_PATH in the CONFIG cell above.")

images_dir = f"data/{SCENE}/images"
os.makedirs(images_dir, exist_ok=True)

existing = sorted(glob.glob(images_dir + "/*.png"))
if existing:
    print(f"{len(existing)} frames already in {images_dir} -- skipping extraction "
          f"(delete the folder to re-extract).")
else:
    # Copy video Drive -> local disk ONCE, then decode locally. ffmpeg reading off
    # the Drive FUSE mount is a network round-trip per read (~0.2 it/s lesson).
    local_video = "/content/" + os.path.basename(VIDEO_PATH)
    if not os.path.exists(local_video):
        print("copying video Drive -> local disk (one-time)...")
        shutil.copy2(VIDEO_PATH, local_video)
    # NATIVE resolution: fps filter ONLY (no scale) -> no upscale, no aspect stretch.
    subprocess.run([
        "ffmpeg", "-y", "-loglevel", "error",
        "-i", local_video,
        "-vf", f"fps={EXTRACT_FPS}",
        "-q:v", "1",
        f"{images_dir}/frame_%04d.png",
    ], check=True)
    existing = sorted(glob.glob(images_dir + "/*.png"))

n = len(existing)
print(f"extracted {n} frames -> {images_dir}")
if n < 100:
    print(f"  WARNING: {n} frames is too sparse -- raise EXTRACT_FPS (aim 300-800).")
elif n > 800:
    print(f"  WARNING: {n} frames -- the COLMAP mapper will be slow (~2-4 h); "
          f"consider lowering EXTRACT_FPS.")
else:
    print(f"  OK: {n} frames (no warning; 300-800 is the sweet spot).")

## 6. Train — `ultra` + foundation + native-res

`ultra` trains **with** `--foundation` (the Metric3D depth prior — max quality) and
**always** with `--native-res`, which derives the train/eval resolution from the extracted
frames and clamps `ultra`'s 3840 multires finale down to the true source long edge (so the
finale never upsamples). `--colmap-cpu` is added **only** when the CUDA wrapper is absent
(bootstrap fell back to apt CPU SIFT); `--nvs-eval` is added only when `RUN_EVAL` is True.

In [ ]:
import os, time

# --foundation: ultra uses the Metric3D depth prior (max quality).
# --colmap-cpu ONLY when the CUDA wrapper is absent (bootstrap fell back to apt CPU SIFT).
# --native-res ALWAYS: ultra is built for it -- train/eval at the frames' native res and
#   clamp the 3840 multires finale down to the true source long edge (no upsampling).
# --nvs-eval ONLY when RUN_EVAL (holds out every 8th frame for metrics).
_colmap = "" if os.path.exists("/usr/local/bin/colmap") else "--colmap-cpu "
_eval   = "--nvs-eval " if RUN_EVAL else ""
_cmd = (f"python scripts/static_3dgs.py --scene {SCENE} --preset {PRESET} "
        f"--foundation {_eval}{_colmap}--native-res").strip()
print("COLMAP:", "CUDA wrapper (GPU SIFT)" if not _colmap else "apt CPU fallback (--colmap-cpu)")
print("EVAL  :", "held-out every-8" if RUN_EVAL else "OFF (all frames train)")
print("RUN   :", _cmd)

RUN_START = time.time()   # freshness anchor for the metrics cell
!{_cmd}

## 7. Metrics — only meaningful when `RUN_EVAL`

With `RUN_EVAL=True` this prints the held-out PSNR/SSIM/LPIPS (every 8th frame). There is no
`sota_compare` here — a personal scene has no published baseline, so the numbers are only
for your own comparison. With `RUN_EVAL=False` there is nothing to report.

In [ ]:
import sys
sys.path.insert(0, ".")
if RUN_EVAL:
    from colab.verify_helpers import assert_fresh_eval, read_metrics
    # A crashed run leaves the PREVIOUS nvs_eval.json in place; refuse a stale file.
    assert_fresh_eval(SCENE, RUN_START)
    m = read_metrics(SCENE)
    print(f"held-out PSNR : {m['psnr']:.2f} dB")
    print(f"SSIM / LPIPS  : {m['ssim']:.4f} / {m['lpips']:.4f}  [{m['kind']}, n={m['n_frames']}]")
else:
    print("eval skipped -- all frames went to training (RUN_EVAL=False). "
          "Set RUN_EVAL=True for held-out PSNR/SSIM/LPIPS.")

## 8. Save to Drive — splat (always) + results + walkable world

The splat `.ply` is copied **first and unconditionally**, so you keep it even if the
world-wrap step errors. Drive is already mounted (cell 2), so these just call the helpers.

In [ ]:
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import save_splat_to_drive, copy_results_to_drive, wrap_and_save_world
# Drive is already mounted (cell 2); drive.mount is idempotent, so no remount here.

# (a) raw splat .ply -- saved FIRST, always (open in any 3DGS viewer)
save_splat_to_drive(SCENE, "/content/drive/MyDrive/4dgs")
# (b) full results (eval json, logs, orbit.mp4)
copy_results_to_drive(SCENE, "/content/drive/MyDrive/4dgs/results")
# (c) walkable world bundle -- best effort; the splat above is safe even if this fails
try:
    wrap_and_save_world(SCENE, "/content/drive/MyDrive/4dgs")
except Exception as e:
    print("  world wrap skipped:", e, "\n  -> the splat .ply above is still on Drive")

**View it & walk it.**

- *Splat only* (simplest): open `MyDrive/4dgs/splats/myroom-max.ply` in any 3DGS viewer,
  e.g. https://superspl.at/editor — no project needed.
- *Walkable*: download `MyDrive/4dgs/worlds/<SCENE>/` into your local `worlds/<SCENE>/`, run
  `cd frontend && npm run dev`, open the **Interactive** page, and pick the world in the
  top-right dropdown.

**Heads-up on the dropdown:** the viewer's world list is **hardcoded** in
`frontend/src/interactive/WorldSelector.tsx` and has **no `myroom-max` entry**. So either
(a) rename the downloaded folder to `worlds/myroom/` — the dropdown already has a **My Room**
slot pointing at `worlds/myroom/output/world/...` — or (b) ask for a selector entry to be
added for your `SCENE` slug. Without one of those two, the dropdown will not list your world.